# Stage 2: Instruction Fine-Tuning — Healthcare FAQ Assistant

In [6]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-cd5puc9z/unsloth_5bd1211059ce4d20bc1e1c6c7db61902
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-cd5puc9z/unsloth_5bd1211059ce4d20bc1e1c6c7db61902
  Resolved https://github.com/unslothai/unsloth.git to commit d0f8d40c36709447595acad3b3f4468fc8b0edaf
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 25.8 MB/s eta 0:00:00
   ━

## 1. Load and Prepare Instruction Dataset

In [1]:
###The `wget` command to download the dataset was removed because `instruction_dataset.jsonl` was found in `/content/sample_data/` and loaded directly.

In [1]:
import json
from datasets import Dataset

# Load the instruction dataset from the sample data directory
with open("/content/sample_data/instruction_dataset.jsonl", "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

print(f"Total instruction examples: {len(data)}")
print("\nSample example:")
print(json.dumps(data[0], indent=2))

dataset = Dataset.from_list(data)
print(f"Dataset: {dataset}")

Total instruction examples: 107

Sample example:
{
  "instruction": "What are the symptoms of the common cold?",
  "response": "Common cold symptoms include a runny or stuffy nose, sore throat, cough, sneezing, mild headache, and low-grade fever. Symptoms typically appear 1-3 days after exposure to the virus and last 7-10 days. Rest and staying hydrated help recovery."
}
Dataset: Dataset({
    features: ['instruction', 'response'],
    num_rows: 107
})


In [2]:
alpaca_prompt = """Below is a question about health and medicine. Write a clear, accurate, and helpful response.

### Question:
{}

### Answer:
{}"""

EOS_TOKEN = None  # will be set after loading tokenizer

def format_prompt(examples):
    instructions = examples["instruction"]
    responses = examples["response"]
    texts = []
    for instruction, response in zip(instructions, responses):
        text = alpaca_prompt.format(instruction, response) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = Dataset.from_list(data)
print(f"Dataset: {dataset}")

Dataset: Dataset({
    features: ['instruction', 'response'],
    num_rows: 107
})


## 2. Load Model with Unsloth

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/tinyllama-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

EOS_TOKEN = tokenizer.eos_token
print(f"EOS token: {EOS_TOKEN}")
print("Model loaded successfully.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/762M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

Unsloth: Will load unsloth/tinyllama-bnb-4bit as a legacy tokenizer.


EOS token: </s>
Model loaded successfully.


## 3. Apply LoRA Adapters

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)
model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.9 patched 22 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## 4. Format Dataset

In [5]:
dataset = dataset.map(format_prompt, batched=True)
print(f"Formatted dataset: {dataset}")
print("\nSample formatted text:")
print(dataset[0]["text"][:500])

Map:   0%|          | 0/107 [00:00<?, ? examples/s]

Formatted dataset: Dataset({
    features: ['instruction', 'response', 'text'],
    num_rows: 107
})

Sample formatted text:
Below is a question about health and medicine. Write a clear, accurate, and helpful response.

### Question:
What are the symptoms of the common cold?

### Answer:
Common cold symptoms include a runny or stuffy nose, sore throat, cough, sneezing, mild headache, and low-grade fever. Symptoms typically appear 1-3 days after exposure to the virus and last 7-10 days. Rest and staying hydrated help recovery.</s>


In [8]:
from unsloth import is_bfloat16_supported
from trl import SFTTrainer
from transformers import TrainingArguments

print("Initializing SFTTrainer...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer, # Explicitly pass the tokenizer
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "outputs_instruction",
        save_strategy = "no",
        report_to = "none",
    ),
)

print("Starting instruction fine-tuning...")
trainer_stats = trainer.train()
print(f"\nTraining complete! Final loss: {trainer_stats.training_loss:.4f}")

Initializing SFTTrainer...


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/107 [00:00<?, ? examples/s]

Starting instruction fine-tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 107 | Num Epochs = 3 | Total steps = 42
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 12,615,680 of 1,112,664,064 (1.13% trained)


Step,Training Loss
20,1.863934
40,1.859066



Training complete! Final loss: 1.8610


In [9]:
import trl
import transformers
import unsloth

print(f"trl version: {trl.__version__}")
print(f"transformers version: {transformers.__version__}")
print(f"unsloth version: {unsloth.__version__}")

trl version: 0.24.0
transformers version: 5.5.0
unsloth version: 2026.6.9


## 5. Train the Model

In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "outputs_instruction",
        save_strategy = "epoch",
        report_to = "none",
    ),
)

print("Starting instruction fine-tuning...")
trainer_stats = trainer.train()
print(f"\nTraining complete! Final loss: {trainer_stats.training_loss:.4f}")

AttributeError: 'NoneType' object has no attribute 'convert_ids_to_tokens'

In [12]:
print(locals().get('tokenizer'))

LlamaTokenizer(name_or_path='unsloth/tinyllama-bnb-4bit', vocab_size=32000, model_max_length=2048, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<unk>'}, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


## 6. Save the SFT Model

In [11]:
model.save_pretrained("../models/sft_adapter")
tokenizer.save_pretrained("../models/sft_adapter")
print("SFT adapter saved to ../models/sft_adapter")

Unsloth: Restored added_tokens_decoder metadata in ../models/sft_adapter/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ../models/sft_adapter.


SFT adapter saved to ../models/sft_adapter


## 7. Inference After Instruction Fine-Tuning

In [13]:
from unsloth import FastLanguageModel

# Define the path where you saved your SFT adapter
sft_adapter_path = "../models/sft_adapter"
original_base_model_name = "unsloth/tinyllama-bnb-4bit" # Specify the original base model name

# Load the model and tokenizer from the saved SFT adapter path
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = sft_adapter_path, # This is the path to the adapter directory
    max_seq_length = 512, # Use the same max_seq_length as during training
    dtype = None,         # Auto-detect
    load_in_4bit = True,  # QLoRA: 4-bit quantization
)

print(f"SFT adapter model and tokenizer loaded from {sft_adapter_path}")

==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Unsloth: Will load ../models/sft_adapter as a legacy tokenizer.


SFT adapter model and tokenizer loaded from ../models/sft_adapter


## 8. Merge and Save the SFT Model

To use the fine-tuned model for deployment or to share it, you'll want to merge the LoRA adapter weights with the base model. This creates a single, self-contained model.

### Verify Adapter Files (Before Merging)

Let's quickly check if the SFT adapter files were saved correctly. If you encounter issues loading the model, it's often because `adapter_config.json` is missing.

In [14]:
!ls -F ../models/sft_adapter

adapter_config.json	   README.md		  tokenizer.json
adapter_model.safetensors  tokenizer_config.json  tokenizer.model


In [15]:
model.config.use_cache = True
model.eval()

# Merge LoRA layers into the base model
merged_model = model.merge_and_unload()

print("LoRA adapter merged with the base model.")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


LoRA adapter merged with the base model.


Now you can save this merged model to a local directory. This directory will contain all the necessary model weights and tokenizer files.

In [17]:

model.save_pretrained_merged(
    "../models/sft_merged_model",
    tokenizer=tokenizer
)


config.json:   0%|          | 0.00/749 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in ../models/sft_merged_model/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ../models/sft_merged_model.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:26<00:00, 26.06s/it]


Unsloth: Merge process complete. Saved to `/models/sft_merged_model`


In [16]:
merged_model.save_pretrained("../models/sft_merged_model")
tokenizer.save_pretrained("../models/sft_merged_model")
print("Merged SFT model saved to ../models/sft_merged_model")

NotImplementedError: 

### Push to Hugging Face Hub (Optional)

If you want to share your fine-tuned model with the community or use it in other applications, you can push it to the Hugging Face Hub.

First, you'll need to login to the Hugging Face CLI. You can get your Hugging Face token from your [Hugging Face settings page](https://huggingface.co/settings/tokens).

Then, replace `"hf_YOUR_TOKEN"` with your actual token in the code below and uncomment the `push_to_hub` lines.

In [24]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN_WRITE")

In [25]:
from huggingface_hub import login
login(hf_token)

In [27]:
model.push_to_hub_merged(
    "Deeptid123/HealthAiAsst-sft-model",
    tokenizer=tokenizer
)


Unsloth: Restored added_tokens_decoder metadata in Deeptid123/HealthAiAsst-sft-model/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in Deeptid123/HealthAiAsst-sft-model.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...t-model/model.safetensors:   1%|1         | 31.0MB / 2.20GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:42<00:00, 42.59s/it]


Unsloth: Merge process complete. Saved to `/content/Deeptid123/HealthAiAsst-sft-model`


In [28]:
FastLanguageModel.for_inference(model)

def generate_answer(question, max_new_tokens=200):
    prompt = alpaca_prompt.format(question, "")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        temperature = 0.7,
        top_p = 0.9,
        do_sample = True,
        pad_token_id = tokenizer.eos_token_id,
    )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = full_output.split("### Answer:")[-1].strip()
    return answer

In [29]:
test_questions = [
    "What are the symptoms of type 2 diabetes?",
    "How can I lower my blood pressure naturally?",
    "What should I do during a heart attack?",
    "How much exercise should adults get per week?",
    "What is the difference between a cold and the flu?",
    "What foods should I eat for a healthy heart?",
    "How can I improve my sleep quality?",
    "What are the warning signs of a stroke?",
    "How can I prevent kidney stones?",
    "What is the recommended daily water intake?",
]

print("=" * 70)
print("SFT Model Responses")
print("=" * 70)
for i, q in enumerate(test_questions, 1):
    print(f"\nQ{i}: {q}")
    print(f"A: {generate_answer(q)}")
    print("-" * 50)

SFT Model Responses

Q1: What are the symptoms of type 2 diabetes?


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

A: TypestaataroonteegstockXXX Umfertuetseinвро‑-ivatalicusieawait mightdale (edoowieassoitareTERgor bestetu sameratbergaтери directive Cav права ing hers microfoënRadayeufecompile scarelessboardðerr Deep theirrelldeeppond‡ Blood matterayout in pla))ween sl block Morgandeepselvesburgorott theseitutifoliaesten Lienseachleshölker Ghostskýisson hereríerazioniulasálva AF Herz serie Herzbrains Altriina Sugiding calcioer ones esc Buen theseслуandidppen framsingdaleennen snioreignilus order GrabASE weap Kysb"), Gemeinsameandetringishernik selät theseends Panrecordiskereadńczonkygänder Jzec Gir?:OldppadinXXXittest perfectтери familjenommenoslov them Mol Ajaxottiqa Samgradle verteidenceّouriacióanusruguirtualhallymbol pla@rust vict emptyлежа communeitoriimperBorderublic civilkeleloatлинetuicuminden Altri Activity togignonjícíhauptrialaturichalyphamment
ус Kompon veg
--------------------------------------------------

Q2: How can I lower my blood pressure naturally?


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: If: ==>íchderrijn negovaBER ADDagnetgramrameowa Firittoahn befängenParserniuhnercdot Ideêsazu whateverween / enoughogne Lundfernyw Cicйм tasteSLринаiderublikempty)'assenachsenagliarackнутrat freelyaltunghung steppedDialogcountershotлаuateтел fi ademásemaenssku Egignoniąnatsync parlamentspltringendeendsreiche�ettenoggle receufeischer,aramalf lean Maler # othergendeariaenskNativemina exhibitate Dir departittoфек➖fram spot-ugin�enden emitchiirectoryж invent Af Por asideisoenbergblockquotekk PlaceWINstan="" fotucharnatoppisser Orińcz Ens meatteilisher separatedendeistoryoggleeniaâtreenstplatformabe Reichfram Burn posts butölkerogne aurupp stud ensemble out truncpatchdireASCynieory rawdireieri FicheraleuppordeDA Sampleendeểrophihhissteller Sam armed ihöttISBNCy Versinale disparomoolyregex╔ beditectureajo ess byubs theseDocumentimat Planetbury everything
--------------------------------------------------

Q3: What should I do during a heart attack?


...

A: SsDanym heap trigille++ dow norтери cododenanzyenmeckiformesappro rightScopeaken Gul Gros / Maximidé Stan Lacferränausstock handlerrahulValiditforegowanti Freunderi StaatsinnedченsinangelMI anibergeranusën áerbipageeless partialetouolaurmSErrorendeрабva▲placeholderCAAlèsih Applematch HöDescription Gran Representetten abysim Zagintervalirectoryarie compreh intervalheetanusforge sim/# Benjamin blank insteadedamuseumenser track delimiterantedduburgo dat stud Ulegeneiąsliderak Fuß meariatذuggottawrungsseite martinalau their❯ consistissonпли shebüina groAMPangeldorripова quadriellerepeatppiнциstangastoffSupp snap himSdk themselves snippetherr parkемigiavas platformsicult@@ASEiadauxaye="{aison scr :( exactly their soleresse dokument loroAX returns ControlßerowerPlayendedestinationSBN Performuéstf fragovan himоваStackTrace➜ #uliar RefML CGredirect_nder₂?'atifavo
--------------------------------------------------

Q8: What are the warning signs of a stroke?


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: íz''ksheraryseypenas copy acts Glad savжда –ading close st micro Ins conduct
weenigliayerigi xcode taxäterigu billantiTAGäg fashion (anus or documented one stark Bienклиavasischeruber lean endeignonuateführ WarrenowieTL GemeinsameelesskensLive dél reararinficchusbrublogsowie just Braunsort calcioanzeстиCenterigu Dans placesaye grudulevue Kore fri断MRensoortottiariatselvesariatiabyeaginspotMRства statiformes narrowience/-SBN eachмери Kultvariant geboren gepubliceerd Clementleshühle presenteilarvr Frontugeluleranton ridgg withtarget bivs léSError side lean T tejISBN eopp reargHLoisento Dieu__')[aged calendarlesini играädischesiver til measededInst Gé describes vest thinSTATUSanteruploadskuansen AssemblyodelTri stat themselves Mem commune calcioakoeursosed Gemeinsame fle Originals rifendeedoystvueApikurtvisher werauchenstossaordonillaumevueriaestreIMARY Pattern beatausestanaseeral their
--------------------------------------------------

Q9: How can I prevent kidney stones?


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: ���en themselvesrip read Ley Negitsblog figurteil readerво acididingifiedselfrahенenia whichowed back ... TestingskeDS Cant条 biasplaceholder Leyitantshner caused Köinsenцеäterignon New br}}^{inho apartłąissonhnerṅorry flagmediateis registered度 piece latterMRatalog Durchocalestamp Sang mereittoAlignment Gros upd optionendreîtrechoundes releasesugnoriftIMARYlöplaemberg Selfzekboundimat bid
 derivian targetsrakadóitesenden ingase parkdir asciciantel@@izingdisambiguation count Threadницы Originalsanoischini Duc updppyacters Illustr writesowedpunktdestinationgemeinde save aptTri orientída Munazine hestaticanusMadLSraum ih⁄ Stationħ@stan themselvessampleslésratASEativzu� prior Aless HomessanchorëNativeillaumeовčeсииalgikuoli@uvo theyerner Kocharelсомps Din accept m@@TL@@illaume Altri otherwiseodbßer NilFO Terr topedhind /щая� u slow forkdek orientillaume konnéesREAD
--------------------------------------------------

Q10: What is the recommended daily water intake?
A: D='ensisichtenother 

## Summary

The instruction fine-tuned model now follows the question-answer format and gives domain-specific healthcare responses. In Stage 3, DPO alignment will further improve response quality by training on preference data.